# IE212 – Lab 2: Spark RDD
## Notebook thực hành trên lớp (đã hoàn thành)

Notebook gồm Phần A (chuẩn bị), Phần B (RDD cơ bản), Phần C (TransAnalysis 1–6) và Phần D (MovieLens 1–5).

- Toàn bộ 26 TODO trong Phần C và D đã được điền. Chạy các ô theo thứ tự từ trên xuống.
- Chú thích `# phải ra ...` là kết quả mẫu để tự đối chiếu với dữ liệu gốc.
- Sử dụng RDD API trong các bài thực hành, không xử lý dữ liệu qua pandas/DataFrame.

### Dữ liệu

- `data/movielens-mini`: `movies.txt`, `ratings_1.txt`, `ratings_2.txt`, `users.txt`, `occupation.txt`.
- `data/trans`: `trans240_1.txt`, `trans240_2.txt`, `cust.txt`.

Trên Google Colab: chuẩn bị `data.zip` chứa thư mục `data/` và chạy ô tải dữ liệu ở Phần A.


---
# PHẦN A. CHUẨN BỊ TRƯỚC BUỔI HỌC

## A.1. Môi trường thực hành

### Bước 1. Tạo notebook Colab và cài PySpark

Mở colab.research.google.com, tạo notebook mới rồi chạy ô dưới.
Máy cá nhân đã cài sẵn PySpark thì bỏ qua ô này.

In [1]:
!pip install -q pyspark==3.5.1

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.1 which is incompatible.


### Bước 2. Nạp dữ liệu của lab lên Colab

Nén thư mục `data/` của Lab 2 thành `data.zip` rồi chạy ô dưới.
Ô này chỉ chạy trên Colab, trên máy cá nhân nó tự bỏ qua.

In [2]:
import os
import zipfile

TREN_COLAB = os.path.isdir("/content")

if TREN_COLAB:
    data_root = "/content/data"
    required = [
        "movielens-mini/movies.txt", "movielens-mini/ratings_1.txt",
        "movielens-mini/ratings_2.txt", "movielens-mini/users.txt",
        "movielens-mini/occupation.txt", "trans/trans240_1.txt",
        "trans/trans240_2.txt", "trans/cust.txt"
    ]
    if not all(os.path.isfile(os.path.join(data_root, f)) for f in required):
        from google.colab import files
        print("Hay chon file data.zip chua hai thu muc data/movielens-mini va data/trans")
        uploaded = files.upload()
        if "data.zip" not in uploaded and not os.path.isfile("data.zip"):
            raise FileNotFoundError("Can upload file data.zip")
        with zipfile.ZipFile("data.zip") as z:
            z.extractall("/content/")

    missing = [f for f in required if not os.path.isfile(os.path.join(data_root, f))]
    if missing:
        raise FileNotFoundError("data.zip thieu file hoac sai cau truc: " + ", ".join(missing))
    print("Da nap du 8 tep du lieu:", sorted(os.listdir(data_root)))
else:
    print("Khong phai Colab, bo qua buoc upload data.zip.")


Hay chon file data.zip chua hai thu muc data/movielens-mini va data/trans


Saving data.zip to data.zip
Da nap du 8 tep du lieu: ['movielens-mini', 'openflights', 'trans']


### Bước 3. Tạo SparkSession

SparkSession là điểm vào của mọi ứng dụng Spark từ bản 2.0. Từ nó lấy ra
`sparkContext` để làm việc với RDD ở Phần B và Phần C.

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("IE212-Lab2") \
    .master("local[*]") \
    .getOrCreate()

# master("local[*]") nghĩa là chạy tại chỗ, dùng hết số lõi CPU đang có.
# sparkContext là điểm vào từ thời Spark 1.x, nay vẫn cần cho RDD API.
sc = spark.sparkContext

# Bớt log INFO cho dễ nhìn khi chiếu lên màn hình.
sc.setLogLevel("ERROR")

print(spark.version)        # 3.5.1

3.5.1


### Bước 4. Tuỳ chọn: chạy trên máy cá nhân

Dành cho ai muốn chạy offline để xem được Spark UI ở `localhost:4040`.
Hai lệnh dưới chạy trong terminal, không phải trong notebook:

```
pip install pyspark==3.5.1
java -version
```

Dòng đầu của `java -version` phải có dạng `openjdk version "11.0.x"` hoặc `"17.0.x"`.
JDK 21 trở lên sẽ làm Spark 3.5 ném `UnsupportedClassVersionError` ngay khi tạo
SparkSession. Windows còn cần `winutils.exe` và `hadoop.dll` bản Hadoop 3.3.x.

Nếu máy bạn đã cài sẵn một bản Spark khác và có biến `SPARK_HOME`, hãy xoá biến đó
trước khi chạy, vì PySpark sẽ ưu tiên dùng jar của bản cũ và ném
`ClassNotFoundException`.

## A.2. Dữ liệu của lab

Bốn tệp MovieLens trong `data/movielens-mini`, phân tách bằng dấu phẩy,
không có dòng tiêu đề:

```
movies.txt      : MovieID, Title, Genres          (Genres phân tách bằng "|")
ratings_1/2.txt : UserID, MovieID, Rating, Timestamp
users.txt       : UserID, Gender, Age, Occupation, Zip-code
occupation.txt  : ID, Occupation
```

Ba tệp giao dịch trong `data/trans`, cũng phân tách bằng dấu phẩy và không có
dòng tiêu đề. Đây đúng là bộ dữ liệu của Lab 1:

```
trans240_1/2.txt : TransID, Date, CustID, Cost, Game,
                   Equipment, City, State, Mode
cust.txt         : CustID, FirstName, LastName, Age, Profession
```

Bộ MovieLens này không có dấu cách sau dấu phẩy, khác bộ của Lab 1, nên `int()`
chạy thẳng được mà không cần cắt khoảng trắng. Đổi lại, cột `Title` nằm giữa nên
khi tách vẫn phải viết `split(",", 2)` chứ không phải `split(",")`.

In [4]:
# Xac dinh thu muc du lieu cho Google Colab hoac may ca nhan.
if TREN_COLAB:
    DATA = "/content/data"
else:
    from pathlib import Path
    candidates = [Path.cwd() / "data", Path.cwd().parent / "data"]
    DATA = str(next((p for p in candidates if (p / "movielens-mini").is_dir() and (p / "trans").is_dir()), candidates[0]))

ML = os.path.join(DATA, "movielens-mini")
TR = os.path.join(DATA, "trans")
print(ML, sorted(os.listdir(ML)))
print(TR, sorted(os.listdir(TR)))


/content/data/movielens-mini ['movies.txt', 'occupation.txt', 'ratings_1.txt', 'ratings_2.txt', 'users.txt']
/content/data/trans ['cust.txt', 'trans240_1.txt', 'trans240_2.txt']


---
# PHẦN B. LÀM QUEN VỚI SPARK RDD

Phần B dựng phiên làm việc và ba khái niệm nền của RDD API: transformation,
action và lazy evaluation. Bốn RDD đọc ở Bước 3 tới Bước 5 là của bộ MovieLens
và chỉ dùng tới ở Phần D, đọc sẵn ngay đây để về sau không phải quay lại.

## B.1. RDD, transformation và action

### Bước 1. Tạo RDD đầu tiên và xem số partition

In [5]:
# parallelize() biến một collection của Python thành RDD.
# Tham số thứ hai là số partition muốn chia.
rdd = sc.parallelize([1, 2, 3, 4, 5], 2)

print(rdd.getNumPartitions())   # 2
print(rdd.collect())            # [1, 2, 3, 4, 5]

2
[1, 2, 3, 4, 5]


### Bước 2. Nhìn thấy lazy evaluation

`map` chỉ ghi lại ý định, `collect` mới kích hoạt tính toán.

In [6]:
import time

t0 = time.time()
binh_phuong = rdd.map(lambda x: x * x)          # transformation, chưa tính gì
print("sau map:     %.4f giay" % (time.time() - t0))

t0 = time.time()
print(binh_phuong.collect())                     # action, Spark mới chạy
print("sau collect: %.4f giay" % (time.time() - t0))

# In ra [1, 4, 9, 16, 25].
# Dòng sau map gần như bằng 0. Dòng sau collect mất vài chục tới vài trăm mili-giây.
# Con số tuyệt đối khác nhau theo máy, chênh lệch giữa hai dòng thì luôn thấy được.

sau map:     0.0020 giay
[1, 4, 9, 16, 25]
sau collect: 1.7217 giay


### Bước 3. Đọc dữ liệu MovieLens vào RDD

Hai tệp `ratings_1.txt` và `ratings_2.txt` cùng lược đồ nên nối lại bằng `union`.

In [7]:
# textFile() đọc từng dòng của tệp thành một phần tử kiểu chuỗi.
ratings_raw = sc.textFile(ML + "/ratings_1.txt") \
                .union(sc.textFile(ML + "/ratings_2.txt"))

print(ratings_raw.count())      # 184, bằng 84 cộng 100 dòng
print(ratings_raw.first())      # 7,1020,4.5,1577836800

184
7,1020,4.5,1577836800


### Bước 4. Tách trường và ép kiểu

Dữ liệu đọc vào còn là chuỗi, phải tự tách và tự ép kiểu. Đây là việc mà DataFrame
ở Lab 3 làm giúp bằng một dòng khai báo lược đồ.

In [8]:
def parse_rating(line):
    """'7,1020,4.5,1577836800' thành (7, 1020, 4.5, 1577836800)."""
    uid, mid, rating, ts = line.split(",")
    # Bộ này không có dấu cách sau dấu phẩy nên int() chạy thẳng được.
    # Bộ MovieLens của Lab 1 thì phải cắt khoảng trắng trước.
    return (int(uid), int(mid), float(rating), int(ts))


# cache() vì RDD này được dùng lại ở mọi mục sau.
ratings = ratings_raw.map(parse_rating).cache()

print(ratings.take(3))
# [(7, 1020, 4.5, 1577836800), (23, 1015, 3.5, 1577923200), (45, 1030, 4.0, 1578009600)]

[(7, 1020, 4.5, 1577836800), (23, 1015, 3.5, 1577923200), (45, 1030, 4.0, 1578009600)]


### Bước 5. Đọc ba tệp còn lại

Ba RDD này đều ở dạng cặp `(key, value)` để `join` được với nhau, khoá là cột đầu tiên.

In [9]:
# (MovieID, (Title, Genres))
# split(",", 2) tách tối đa 2 lần nên ra đúng 3 phần, vì cột Title nằm giữa.
movies = sc.textFile(ML + "/movies.txt") \
           .map(lambda l: l.split(",", 2)) \
           .map(lambda p: (int(p[0]), (p[1], p[2])))

# (UserID, (Gender, Age, Occupation)), bỏ cột Zip-code vì không dùng tới.
users = sc.textFile(ML + "/users.txt") \
          .map(lambda l: l.split(",")) \
          .map(lambda p: (int(p[0]), (p[1], int(p[2]), int(p[3]))))

# (OccID, tên nghề)
occupations = sc.textFile(ML + "/occupation.txt") \
                .map(lambda l: l.split(",", 1)) \
                .map(lambda p: (int(p[0]), p[1]))

print(movies.count(), users.count(), occupations.count())   # 50 50 15
print(movies.first())       # (1001, ('The Godfather (1972)', 'Crime|Drama'))

50 50 15
(1001, ('The Godfather (1972)', 'Crime|Drama'))


### Bước 6. Đo tác động của `cache()`

Mỗi action chạy lại toàn bộ chuỗi transformation từ đầu. `cache()` giữ kết quả
lại sau lần tính đầu tiên.

In [10]:
# RDD này cố ý không gọi cache() để làm đối chứng.
khong_cache = sc.textFile(ML + "/ratings_1.txt") \
                .union(sc.textFile(ML + "/ratings_2.txt")) \
                .map(parse_rating)

for lan in (1, 2):
    t0 = time.time()
    n = khong_cache.count()
    print("khong cache, lan %d: %.3f giay" % (lan, time.time() - t0))

for lan in (1, 2):
    t0 = time.time()
    n = ratings.count()          # ratings đã gọi cache() ở Bước 4
    print("co cache,    lan %d: %.3f giay" % (lan, time.time() - t0))

# Cả bốn dòng đều đếm ra 184.
# RDD không cache: lần 2 xấp xỉ lần 1. RDD đã cache: lần 2 nhanh hơn.
# Ở quy mô 184 dòng, phần lớn thời gian là chi phí khởi động job chứ không phải
# xử lý dữ liệu, nên chênh lệch có thể rất nhỏ hoặc đảo chiều ở một lần đo.
# cache() là đánh dấu, không phải action: dữ liệu chỉ nằm trong bộ nhớ sau khi
# có một action chạy qua RDD đó lần đầu.

khong cache, lan 1: 0.325 giay
khong cache, lan 2: 0.260 giay
co cache,    lan 1: 0.328 giay
co cache,    lan 2: 0.201 giay


### Câu hỏi kiểm tra nhanh

1. Vì sao gọi `cache()` xong mà chưa có action nào thì bộ nhớ vẫn chưa giữ gì?
2. RDD nào trong bốn RDD vừa dựng đáng được cache nhất, và vì sao?
3. Vì sao môn học cấm `collect()` toàn bộ dữ liệu rồi xử lý bằng pandas?

---
# PHẦN C. GIẢI LẠI SÁU BÀI TRANSANALYSIS CỦA LAB 1 BẰNG RDD

Sáu bài TransAnalysis của Lab 1 chạy lại trên đúng bộ dữ liệu giao dịch của Lab 1,
lần này bằng RDD. Cùng đề bài, cùng dữ liệu, chỉ khác công cụ. Mọi con số ở đây
đối chiếu được trực tiếp với kết quả MapReduce bạn đã chạy ở Lab 1.

**Độ khó tăng dần trong phần này:**

| Bài | Mức |
|---|---|
| TransAnalysis1 | làm mẫu, code đầy đủ, đọc kỹ rồi theo mẫu này |
| TransAnalysis2 | điền 3 chỗ |
| TransAnalysis3 và 4 | điền 4 chỗ |
| Nạp bảng khách hàng và broadcast | điền 3 chỗ |
| TransAnalysis5 | chỉ còn khung, bạn viết cả biểu thức |
| TransAnalysis6 | trống hoàn toàn, tự viết từ đầu |

## C.1. Giải lại TransAnalysis 1 tới 4 bằng RDD

Bốn bài này chính là TransAnalysis1 tới TransAnalysis4 của Lab 1, chạy trên **đúng bộ
dữ liệu của Lab 1**. Đây là phép so sánh sạch nhất giữa MapReduce và RDD trong cả môn:
cùng đề bài, cùng dữ liệu, chỉ khác công cụ.

```
TransID, Date, CustID, Cost, Game, Equipment, City, State, Mode
0,Jun-26-2011,4000001,40.33,Exercise & Fitness,Cardio Machine Accessories,...
```

Cột `Game` chứa dấu `&` và cột `Equipment` chứa dấu cách, nên chỉ được tách bằng
dấu phẩy. Sau `split(",")`: `p[2]` là CustID, `p[3]` là số tiền, `p[4]` là loại hình
thể thao.

### Bước 1. Đọc dữ liệu giao dịch

In [11]:
# Giữ nguyên mọi cột dưới dạng chuỗi, chỉ ép kiểu khi cần, vì chỉ dùng tới ba cột.
trans = sc.textFile(TR + "/trans240_1.txt") \
          .union(sc.textFile(TR + "/trans240_2.txt")) \
          .map(lambda l: l.split(",")) \
          .cache()

print(trans.count())        # 240 giao dịch của 10 khách hàng
print(trans.first())
# ['0', 'Jun-26-2011', '4000001', '40.33', 'Exercise & Fitness', ...]
#   p[2] là CustID, p[3] là số tiền, p[4] là loại hình thể thao

240
['0', 'Jun-26-2011', '4000001', '40.33', 'Exercise & Fitness', 'Cardio Machine Accessories', 'Clarksville', 'Tennessee', 'credit']


### Bước 2. TransAnalysis1: tổng tiền theo loại hình

**Bài làm mẫu, code đã viết sẵn đầy đủ.** Chạy ô dưới, đối chiếu kết quả, rồi làm
các bài sau theo đúng khuôn này: `map` để phát ra cặp `(khoá, giá trị)`, rồi một
phép gộp theo khoá.

In [12]:
# Phần tính toán chỉ có hai dòng. TransAnalysis1.java của Lab 1 dài 50 dòng.
ta1 = trans.map(lambda p: (p[4], float(p[3]))) \
           .reduceByKey(lambda a, b: a + b)

for game, tong in sorted(ta1.collect()):
    print("%-24s %10.2f" % (game, tong))

# 13 loại hình, tổng cả 13 dòng là 24.652,98. Vài dòng để đối chiếu:
#   Air Sports           142.77
#   Outdoor Recreation  6758.08
#   Winter Sports         98.21
#
# LƯU Ý: không làm tròn thì Spark cho 142.77 còn Lab 1 cho 142.76999759674072.
# Cả hai đều đúng: Java đọc số tiền bằng Float.parseFloat tức float 32 bit rồi mới
# cộng vào biến double, còn Python dùng double suốt từ đầu. Chênh lệch lớn nhất
# trên bộ này là 0,000044. Luôn làm tròn ở bước in.

Air Sports                   142.77
Combat Sports                448.45
Exercise & Fitness          3041.25
Games                        676.89
Gymnastics                  1733.32
Indoor Games                1063.02
Jumping                      894.95
Outdoor Play Equipment      2146.34
Outdoor Recreation          6758.08
Puzzles                      120.29
Team Sports                 3354.95
Water Sports                4174.46
Winter Sports                 98.21


### Bước 3. TransAnalysis2: kèm danh sách CustID

Reducer của Lab 1 phải nhận cả mã khách và số tiền trong một value nên phải nối chuỗi
rồi tách lại. RDD mang được thẳng một tuple.

In [13]:
def gop(vs):
    """vs là các cặp (CustID, Cost) của cùng một loại hình."""
    vs = list(vs)
    return (sorted(v[0] for v in vs), sum(v[1] for v in vs))


# TODO 1: phát ra cặp (loại hình, (CustID, số tiền)).
#   Nhớ ép số tiền sang float, vì textFile đọc vào vẫn là chuỗi.
# TODO 2: gom mọi value cùng khoá vào một danh sách, để hàm gop ở trên xử lý.
#   Gợi ý: đây là phép duy nhất giữ lại được từng phần tử thay vì cộng gộp.
ta2 = trans.map(lambda p: (p[4], (p[2], float(p[3])))) \
           .groupByKey() \
           .mapValues(gop)

for game, (ids, tong) in sorted(ta2.collect()):
    print("%-24s %3d  %10.2f" % (game, len(ids), tong))

# Phải ra đúng 13 dòng, cột giữa cộng lại bằng 240. Vài dòng để đối chiếu:
#   Air Sports                 4      142.77
#   Outdoor Recreation        45     6758.08
#   Winter Sports              8       98.21
#
# Outdoor Recreation có 45 mã trong khi chỉ có 6 khách hàng khác nhau, vì mỗi giao
# dịch góp một mã kể cả khi trùng. Bước 4 xử lý chuyện này.

Air Sports                 4      142.77
Combat Sports              7      448.45
Exercise & Fitness        32     3041.25
Games                      8      676.89
Gymnastics                17     1733.32
Indoor Games              13     1063.02
Jumping                   10      894.95
Outdoor Play Equipment    17     2146.34
Outdoor Recreation        45     6758.08
Puzzles                    4      120.29
Team Sports               32     3354.95
Water Sports              43     4174.46
Winter Sports              8       98.21


### Bước 4. TransAnalysis3 và TransAnalysis4: khử trùng lặp

Ở Lab 1, hai bài này cần một `List` trong Reducer cộng với `contains()` để lọc mã trùng.
Trên RDD chỉ cần đưa về cặp `(Game, CustID)` rồi gọi `distinct()`, và cả hai bài dùng
chung kết quả đó.

In [14]:
# TODO 3: đưa dữ liệu về cặp (loại hình, CustID) rồi bỏ các cặp trùng nhau.
#   RDD này được dùng lại ba lần ở dưới và cả ở C.2, nên nhớ cache().
cap = trans.map(lambda p: (p[4], p[2])).distinct().cache()
print(cap.count())              # phải ra 44

# TODO 4: TransAnalysis3. Với mỗi loại hình, lấy danh sách CustID đã sắp xếp.
#   Gợi ý: gom theo khoá rồi sắp xếp phần value.
ta3 = cap.groupByKey().mapValues(lambda vs: sorted(vs))

# TODO 5: TransAnalysis4. Đếm số người chơi khác nhau của mỗi loại hình.
#   Gợi ý: vì cap đã hết trùng, mỗi phần tử đáng giá đúng 1. Biến value thành 1
#   rồi cộng lại theo khoá.
ta4 = cap.mapValues(lambda _: 1).reduceByKey(lambda a, b: a + b)

tong = ta1.collectAsMap()       # dùng lại kết quả Bước 2
dem = ta4.collectAsMap()

for game, ids in sorted(ta3.collect()):
    print("%-24s %d  [%s]  %10.2f"
          % (game, dem[game], ",".join(ids), tong[game]))

# Phải ra 13 dòng. Vài dòng để đối chiếu:
#   Air Sports               1  [4000005]                      142.77
#   Combat Sports            2  [4000001,4000009]              448.45
#   Winter Sports            3  [4000001,4000003,4000006]       98.21
#
# Cột số người chơi phải trùng với số phần tử trong ngoặc vuông ở từng dòng.
# Cột tổng tiền giữ nguyên như Bước 2, vì tổng tiền vẫn cộng mọi giao dịch kể cả
# của mã trùng. Đối chiếu cả 13 dòng với TransAnalysis3 và 4 của Lab 1.

44
Air Sports               1  [4000005]      142.77
Combat Sports            2  [4000001,4000009]      448.45
Exercise & Fitness       5  [4000001,4000002,4000005,4000007,4000010]     3041.25
Games                    2  [4000008,4000010]      676.89
Gymnastics               4  [4000001,4000003,4000009,4000010]     1733.32
Indoor Games             2  [4000004,4000009]     1063.02
Jumping                  2  [4000006,4000010]      894.95
Outdoor Play Equipment   4  [4000005,4000006,4000008,4000009]     2146.34
Outdoor Recreation       6  [4000001,4000002,4000003,4000004,4000007,4000008]     6758.08
Puzzles                  1  [4000005]      120.29
Team Sports              5  [4000002,4000005,4000007,4000008,4000010]     3354.95
Water Sports             7  [4000001,4000002,4000003,4000004,4000006,4000008,4000009]     4174.46
Winter Sports            3  [4000001,4000003,4000006]       98.21


### Câu hỏi kiểm tra nhanh, mục C.1

1. Vì sao `cap` được gọi `cache()`? Nó được dùng mấy lần trong Bước 4?
2. Thay `distinct()` bằng `groupByKey()` rồi `set()` trong `mapValues` thì kết quả
   có đổi không? Còn khối lượng shuffle thì sao?
3. `contains()` của Lab 1 chậm ở đâu khi một loại hình có vài triệu khách hàng?
   `distinct()` của Spark khác ở chỗ nào?
4. `ta1.collectAsMap()` kéo kết quả về driver. Với 13 loại hình thì không sao.
   Khi nào cách này thành nguy hiểm?

## C.2. Giải lại TransAnalysis 5 và 6 bằng broadcast

Hai bài cuối của Lab 1 cần nối bảng khách hàng để thay mã bằng tên. Vì `cust.txt` chỉ có
10 dòng, Lab 1 dùng Distributed Cache: phát tệp nhỏ về mọi node, mỗi Mapper đọc một lần
trong `setup()` rồi giữ trong `HashMap`. Spark làm đúng việc đó bằng `sc.broadcast`,
không phải viết `setup()`, không phải tự mở tệp.

Lược đồ `cust.txt`:

```
CustID, FirstName, LastName, Age, Profession
4000001,Kristina,Chung,55,Pilot
```

### Bước 1. Nạp bảng khách hàng và broadcast

In [15]:
# TODO 6: biến mỗi dòng cust.txt thành cặp CustID -> (FirstName, Age).
#   Lược đồ: CustID, FirstName, LastName, Age, Profession
#   Chú ý lấy đúng chỉ số cột, LastName không dùng tới.
# TODO 7: kéo kết quả về driver thành một dict Python.
cust = sc.textFile(TR + "/cust.txt") \
         .map(lambda l: l.split(",")) \
         .map(lambda c: (c[0], (c[1], c[3]))) \
         .collectAsMap()

# TODO 8: phát bảng nhỏ này về bộ nhớ của mọi executor.
bc_cust = sc.broadcast(cust)

print(len(bc_cust.value))                  # phải ra 10
print(bc_cust.value["4000001"])            # phải ra ('Kristina', '55')

# collectAsMap() kéo dữ liệu về driver rồi broadcast phát ngược lại cho executor.
# Chỉ dùng được khi bảng nhỏ. Nếu cust.txt có 50 triệu dòng thì vỡ bộ nhớ driver
# ngay ở collectAsMap, lúc đó buộc phải dùng join.

10
('Kristina', '55')


### Bước 2. TransAnalysis5: thay mã bằng tên

Từ đây bạn viết cả biểu thức, chỉ còn vòng lặp in sẵn.

In [16]:
# TODO 9: TransAnalysis5. Đi từ cap là (Game, CustID) đã khử trùng ở Bước 4,
#   đổi mã khách thành tên rồi gom theo loại hình, danh sách tên sắp theo chữ cái.
#
#   Ba phép nối tiếp nhau:
#     1. đổi value từ CustID sang tên, tra bc_cust.value[cid][0]
#     2. gom mọi tên cùng loại hình vào một nhóm
#     3. sắp xếp từng nhóm
#   Phép thứ nhất và thứ ba chỉ đụng vào value nên dùng mapValues, rẻ hơn map.
ta5 = cap.mapValues(lambda cid: bc_cust.value[cid][0]) \
         .groupByKey() \
         .mapValues(lambda vs: sorted(vs))

for game, ten in sorted(ta5.collect()):
    print("%-24s [%s]  %10.2f" % (game, ",".join(ten), tong[game]))

# Tập người chơi phải giống hệt TransAnalysis3 ở C.1, chỉ đổi mã thành tên.
# Vài dòng để đối chiếu:
#   Air Sports         [Karen]                              142.77
#   Combat Sports      [Kristina,Malcolm]                   448.45
#   Winter Sports      [Kristina,Patrick,Sherri]             98.21
# Tên được sắp theo chữ cái nên thứ tự khác bản Java của Lab 1, nhưng tập hợp tên
# thì phải trùng từng dòng.

Air Sports               [Karen]      142.77
Combat Sports            [Kristina,Malcolm]      448.45
Exercise & Fitness       [Dolores,Elsie,Karen,Kristina,Paige]     3041.25
Games                    [Dolores,Hazel]      676.89
Gymnastics               [Dolores,Kristina,Malcolm,Sherri]     1733.32
Indoor Games             [Gretchen,Malcolm]     1063.02
Jumping                  [Dolores,Patrick]      894.95
Outdoor Play Equipment   [Hazel,Karen,Malcolm,Patrick]     2146.34
Outdoor Recreation       [Elsie,Gretchen,Hazel,Kristina,Paige,Sherri]     6758.08
Puzzles                  [Karen]      120.29
Team Sports              [Dolores,Elsie,Hazel,Karen,Paige]     3354.95
Water Sports             [Gretchen,Hazel,Kristina,Malcolm,Paige,Patrick,Sherri]     4174.46
Winter Sports            [Kristina,Patrick,Sherri]       98.21


### Bước 3. TransAnalysis6: tên kèm tuổi

**Bài này bạn tự viết từ đầu, kể cả phần in.**

Đề: với mỗi loại hình thể thao, in danh sách những người đã chơi dưới dạng
`tên,tuổi`, sắp theo chữ cái. Dùng lại `cap` và `bc_cust` đã có.

Gợi ý: giống hệt TransAnalysis5, chỉ khác ở chỗ value là chuỗi ghép tên với tuổi
thay vì chỉ có tên. `bc_cust.value[cid]` vốn đã là cặp `(tên, tuổi)`.

In [17]:
# TODO 10: TransAnalysis6, viết toàn bộ từ đầu.
#
# Kết quả mong đợi, vài dòng để đối chiếu:
#   Air Sports      [Karen,74]
#   Combat Sports   [Kristina,55,Malcolm,39]
#   Winter Sports   [Kristina,55,Patrick,42,Sherri,34]
#
ta6 = cap.mapValues(lambda cid: "%s,%s" % bc_cust.value[cid]) \
         .groupByKey() \
         .mapValues(lambda vs: sorted(vs))

for game, thong_tin in sorted(ta6.collect()):
    print("%-24s [%s]" % (game, ",".join(thong_tin)))

Air Sports               [Karen,74]
Combat Sports            [Kristina,55,Malcolm,39]
Exercise & Fitness       [Dolores,60,Elsie,43,Karen,74,Kristina,55,Paige,74]
Games                    [Dolores,60,Hazel,63]
Gymnastics               [Dolores,60,Kristina,55,Malcolm,39,Sherri,34]
Indoor Games             [Gretchen,66,Malcolm,39]
Jumping                  [Dolores,60,Patrick,42]
Outdoor Play Equipment   [Hazel,63,Karen,74,Malcolm,39,Patrick,42]
Outdoor Recreation       [Elsie,43,Gretchen,66,Hazel,63,Kristina,55,Paige,74,Sherri,34]
Puzzles                  [Karen,74]
Team Sports              [Dolores,60,Elsie,43,Hazel,63,Karen,74,Paige,74]
Water Sports             [Gretchen,66,Hazel,63,Kristina,55,Malcolm,39,Paige,74,Patrick,42,Sherri,34]
Winter Sports            [Kristina,55,Patrick,42,Sherri,34]


### Bước 4. Bảng so sánh MapReduce và RDD

Số dòng bản Java là số dòng không rỗng trong các tệp `TransAnalysis*.java` của Lab 1.
Số dòng bản RDD là phần tính toán, không tính lệnh đọc dữ liệu và lệnh in.

| Bài | Java MapReduce | RDD | Kỹ thuật đổi khác |
|---|:---:|:---:|---|
| TransAnalysis1 | 50 dòng | 2 dòng | `reduceByKey` thay Mapper và Reducer |
| TransAnalysis2 | 58 dòng | 7 dòng | tuple thay việc nối chuỗi rồi tách lại |
| TransAnalysis3 | 63 dòng | 2 dòng | `distinct()` thay `List` và `contains()` |
| TransAnalysis4 | 65 dòng | 2 dòng | dùng lại chính RDD `cap` của bài trước |
| TransAnalysis5 | 96 dòng | 4 dòng | `sc.broadcast` thay Distributed Cache và `setup()` |
| TransAnalysis6 | 96 dòng | 3 dòng | như trên, không còn bẫy lệch chỉ số cột |

Viết ba tới năm câu nhận xét vào báo cáo. Ba điểm đáng nói: RDD bỏ được toàn bộ
phần khai báo lớp Mapper, lớp Reducer và hàm `main`; nhiều bài dùng lại được kết quả
trung gian của bài trước thay vì viết lại từ đầu; nhưng RDD vẫn để bạn tự quản lý
việc khử trùng và tự chọn giữa `join` với `broadcast`, những việc mà DataFrame ở
Lab 3 sẽ làm thay.

### Câu hỏi kiểm tra nhanh, mục C.2

1. Bỏ `broadcast` đi, tra cứu thẳng vào biến `cust` của driver thì chương trình vẫn
   chạy đúng. Vậy `broadcast` có lợi ở đâu?
2. TransAnalysis6 của Lab 1 ra sai cột tổng tiền mà job vẫn báo thành công. Làm thế
   nào phát hiện được loại lỗi đó khi không có con số biết trước để đối chiếu?
3. Trong sáu bài trên, bài nào sinh shuffle và bài nào không?

---
# PHẦN D. PHÂN TÍCH DỮ LIỆU MOVIELENS BẰNG RDD



## D.1. Bài 1. Điểm trung bình và số lượt đánh giá của từng phim

Đề bài: tính điểm trung bình và tổng số lượt đánh giá của từng phim từ cả hai tệp
ratings, sau đó tìm phim có điểm trung bình cao nhất trong số các phim đạt tối thiểu
5 lượt đánh giá.

### Bước 1. Gộp theo MovieID

RDD không có phép tính trung bình theo khoá. Cách làm chuẩn là mang theo cặp
`(tổng_điểm, số_lượt)` qua `reduceByKey` rồi chia ở bước cuối.



In [18]:
# (MovieID, (tổng điểm, số lượt))
# Số 1 trong (r[2], 1) là bộ đếm: mỗi bản ghi góp vào đúng 1 lượt.
tong_phim = ratings.map(lambda r: (r[1], (r[2], 1))) \
                   .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))

# mapValues chỉ đụng vào value, giữ nguyên key nên rẻ hơn map.
trung_binh = tong_phim.mapValues(lambda v: (v[0] / v[1], v[1]))

print(trung_binh.count())       # 14, không phải 50
print(trung_binh.take(2))
# movies.txt có 50 phim nhưng chỉ 14 phim từng được đánh giá.
# Thứ tự của take(2) phụ thuộc cách chia partition, máy khác có thể ra phim khác.

14
[(1020, (3.6666666666666665, 18)), (1012, (4.0, 2))]


### Bước 2. Nối với bảng phim để lấy tên

Chỗ này thấy rõ cái giá của RDD: phải tự lần theo chỉ số của tuple lồng nhau.

In [19]:
# trung_binh: (MovieID, (diem_tb, so_luot))
# movies:     (MovieID, (Title, Genres))
#
# TODO 11: nối hai RDD trên theo MovieID.
# TODO 12: sau khi nối, mỗi phần tử có dạng
#            (MovieID, ((diem_tb, so_luot), (Title, Genres)))
#                          x[1][0]            x[1][1]
#          Đưa nó về bộ ba (tên phim, điểm trung bình, số lượt).
ket_qua = trung_binh.join(movies) \
                    .map(lambda x: (x[1][1][0], x[1][0][0], x[1][0][1]))

for ten, diem, luot in sorted(ket_qua.collect(), key=lambda x: -x[1]):
    print("%-56s %.2f (%d)" % (ten, diem, luot))

# Phải ra 14 dòng.
# Dòng đầu : Sunset Boulevard (1950)         4.36 (7)
# Dòng cuối: The Silence of the Lambs (1991) 3.14 (7)
# Hai cặp phim bằng điểm tuyệt đối: Psycho và The Godfather: Part II cùng 4.0;
# LOTR Fellowship và No Country for Old Men cùng 3.888889. Thứ tự giữa chúng
# không xác định, chạy lại có thể đảo. Đó không phải lỗi.

Sunset Boulevard (1950)                                  4.36 (7)
The Terminator (1984)                                    4.06 (18)
Psycho (1960)                                            4.00 (2)
The Godfather: Part II (1974)                            4.00 (17)
The Lord of the Rings: The Fellowship of the Ring (2001) 3.89 (18)
No Country for Old Men (2007)                            3.89 (18)
The Social Network (2010)                                3.86 (7)
The Lord of the Rings: The Return of the King (2003)     3.82 (11)
E.T. the Extra-Terrestrial (1982)                        3.67 (18)
Gladiator (2000)                                         3.61 (18)
Fight Club (1999)                                        3.50 (7)
Mad Max: Fury Road (2015)                                3.47 (18)
Lawrence of Arabia (1962)                                3.44 (18)
The Silence of the Lambs (1991)                          3.14 (7)


### Bước 3. Tìm phim có điểm cao nhất, tối thiểu 5 lượt

In [20]:
MIN_RATINGS = 5     # khai báo thành hằng số để đổi ngưỡng cho dễ

# TODO 13: giữ lại các phim có số lượt từ MIN_RATINGS trở lên.
#   ket_qua là bộ ba (tên, điểm, lượt), vậy số lượt nằm ở chỉ số nào?
du_dieu_kien = ket_qua.filter(lambda x: x[2] >= MIN_RATINGS)

# TODO 14: lấy phần tử có điểm trung bình lớn nhất.
#   Gợi ý: có một action nhận tham số key giống hàm max() của Python.
ten, diem, luot = du_dieu_kien.max(key=lambda x: x[1])

print("%s is the highest rated movie with an average rating of %.2f "
      "among movies with at least %d ratings." % (ten, diem, MIN_RATINGS))

# Phải ra Sunset Boulevard (1950) với 4.36.
# Ngưỡng 5 loại ra đúng 1 phim là Psycho (1960) chỉ có 2 lượt, còn 13 phim dự xét.
# Thử đổi MIN_RATINGS = 10: còn 9 phim và quán quân thành The Terminator (1984).

Sunset Boulevard (1950) is the highest rated movie with an average rating of 4.36 among movies with at least 5 ratings.


### Bước 4. So `reduceByKey` với `groupByKey`

In [21]:
# TODO 15: làm lại Bước 1 bằng groupByKey thay vì reduceByKey, để thấy hai cách
#   cho cùng kết quả. Phát ra cặp (MovieID, Rating), gom theo khoá, rồi tính
#   (điểm trung bình, số lượt) từ danh sách value.
#   Gợi ý: trong mapValues, vs là một iterable các điểm đánh giá của cùng một phim.
def thong_ke_diem(vs):
    ds = list(vs)
    return (sum(ds) / len(ds), len(ds))

cach_2 = ratings.map(lambda r: (r[1], r[2])) \
                .groupByKey() \
                .mapValues(thong_ke_diem)

print(sorted(cach_2.collect()) == sorted(trung_binh.collect()))   # phải ra True

# Hai cách cho cùng kết quả, khác nhau ở khối lượng shuffle:
# reduceByKey gộp cục bộ trong từng partition trước, chỉ chuyển 1 cặp số mỗi phim.
# groupByKey chuyển toàn bộ 184 điểm đánh giá.
# Ở quy mô này không thấy gì, với hàng chục triệu bản ghi thì khác hẳn.

True


## D.2. Bài 2. Phân tích đánh giá theo thể loại

Đề bài: một phim thuộc nhiều thể loại cùng lúc, cột Genres ghi chúng cách nhau bằng
dấu `|`. Cần tính điểm trung bình và tổng số lượt đánh giá của từng thể loại.

### Bước 1. Trải phẳng danh sách thể loại bằng `flatMap`

`map` biến 1 phần tử thành đúng 1 phần tử. `flatMap` biến 1 phần tử thành 0, 1
hoặc nhiều phần tử rồi trải phẳng tất cả vào cùng một RDD.

In [22]:
# TODO 16: từ movies là (MovieID, (Title, Genres)), sinh ra một cặp
#   (MovieID, Genre) cho TỪNG thể loại của phim. Cột Genres là chuỗi kiểu
#   "Crime|Drama", nằm ở x[1][1].
#   Gợi ý: một phim sinh ra nhiều bản ghi, nên dùng flatMap chứ không phải map.
the_loai = movies.flatMap(lambda x: [(x[0], g) for g in x[1][1].split("|")])

print(movies.count(), the_loai.count())                      # phải ra 50 122
print(the_loai.filter(lambda x: x[0] == 1001).collect())
# phải ra [(1001, 'Crime'), (1001, 'Drama')]

# 50 phim sinh ra 122 cặp, trung bình mỗi phim thuộc khoảng 2,4 thể loại.
# Ở đây split("|") của Python chạy đúng vì str.split tách theo chuỗi thường.
# Ở Lab 3 sẽ gặp lại cái bẫy này với hàm split của DataFrame, vốn nhận regex.

50 122
[(1001, 'Crime'), (1001, 'Drama')]


### Bước 2. Nối với điểm đánh giá rồi gộp theo thể loại

Từ đây bạn viết cả biểu thức, chỉ còn vòng lặp in sẵn.

In [23]:
# TODO 17: tính điểm trung bình và số lượt của từng thể loại.
#
#   Đường đi gợi ý:
#     1. nối the_loai là (MovieID, Genre) với ratings đã đưa về (MovieID, Rating)
#        kết quả: (MovieID, (Genre, Rating))
#     2. đổi khoá sang Genre và mang theo bộ đếm: (Genre, (Rating, 1))
#     3. cộng gộp theo khoá như Bước 1 của D.1
#     4. chia tổng cho đếm để ra (điểm trung bình, số lượt)
ket_qua_tl = the_loai.join(ratings.map(lambda r: (r[1], r[2]))) \
    .map(lambda x: (x[1][0], (x[1][1], 1))) \
    .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])) \
    .mapValues(lambda v: (v[0] / v[1], v[1]))

for g, (diem, luot) in sorted(ket_qua_tl.collect()):
    print("%-12s %.2f (%d)" % (g, diem, luot))

# Phải ra 12 dòng, từ Action 3.71 (54) tới Thriller 3.70 (27). Vài dòng đối chiếu:
#   Drama        3.76 (128)
#   Film-Noir    4.36 (7)
#   Horror       4.00 (2)
#
# Trên cả 50 phim có 19 thể loại nhưng chỉ ra 12: 7 thể loại còn lại chỉ xuất hiện
# ở những phim không ai đánh giá nên inner join đã loại đi.

Action       3.71 (54)
Adventure    3.63 (83)
Biography    3.56 (25)
Crime        3.81 (42)
Drama        3.76 (128)
Family       3.67 (18)
Fantasy      3.86 (29)
Film-Noir    4.36 (7)
Horror       4.00 (2)
Mystery      4.00 (2)
Sci-Fi       3.73 (54)
Thriller     3.70 (27)


### Bước 3. Kiểm tra lại tổng số lượt

In [24]:
# TODO 18: cộng cột số lượt của cả 12 thể loại lại, rồi in ra cạnh tổng số lượt
#   đánh giá thật sự trong ratings.
print(ket_qua_tl.map(lambda x: x[1][1]).sum())      # phải ra 471
print(ratings.count())                                 # phải ra 184

# 471 chứ không phải 184, và đây không phải lỗi. Giải thích vì sao vào báo cáo.
# Gợi ý: một phim thuộc mấy thể loại thì lượt đánh giá của nó được đếm mấy lần?
# Khi báo cáo phải gọi con số này là lượt đánh giá theo thể loại.

471
184


## D.3. Bài 3. Phân tích đánh giá theo giới tính

Đây là mục trọng tâm của buổi học.

Đề bài: với mỗi phim, tính riêng điểm trung bình của người xem nam và người xem nữ.

Bài này cần hai khoá khác nhau ở hai giai đoạn. Trước hết nối `ratings` với `users`
theo UserID để biết giới tính của người đánh giá. Sau đó đổi khoá sang MovieID để
gộp theo phim. Đổi khoá giữa chừng là thao tác rất hay gặp khi làm việc với RDD cặp.

### Bước 1. Hoàn thiện khung chương trình, phần (1)

In [25]:
# users: (UserID, (Gender, Age, Occupation))
gioi_tinh = users.mapValues(lambda v: v[0])          # (UserID, 'M' hoặc 'F')

# (UserID, (MovieID, Rating))
r_theo_user = ratings.map(lambda r: (r[0], (r[1], r[2])))

# TODO 19, phần (1) của đề: nối r_theo_user với gioi_tinh theo UserID,
#   rồi đổi khoá sang MovieID. Kết quả cần có dạng (MovieID, (Gender, Rating)).
#
#   Sau khi join, mỗi phần tử có dạng (UserID, ((MovieID, Rating), Gender))
#                                                  x[1][0]          x[1][1]
da_noi = r_theo_user.join(gioi_tinh).map(lambda x: (x[1][0][0], (x[1][1], x[1][0][1])))

print(da_noi.count())           # phải ra 184
print(da_noi.take(2))

# Phải đổi khoá giữa chừng vì bài này cần hai khoá ở hai giai đoạn:
# UserID để biết giới tính, rồi MovieID để gộp theo phim.

184
[(1047, ('F', 3.0)), (1012, ('F', 3.5))]


### Bước 1 (tiếp). Hoàn thiện khung chương trình, phần (2) và (3)

In [26]:
# TODO 20, phần (2) của đề: với mỗi MovieID, cộng riêng tổng điểm và số lượt
#   của nam và của nữ.
#   Gợi ý: đưa mỗi bản ghi về vector 4 số (tổng_M, đếm_M, tổng_F, đếm_F) với ba số
#   bằng 0 tuỳ theo giới tính, rồi reduceByKey cộng từng vị trí.
gop = da_noi.map(lambda x: (x[0],
    (x[1][1], 1, 0.0, 0) if x[1][0] == "M" else (0.0, 0, x[1][1], 1))) \
    .reduceByKey(lambda a, b: tuple(a[i] + b[i] for i in range(4)))

print(gop.count())              # phải ra 14


def dinh_dang(v):
    tong_m, dem_m, tong_f, dem_f = v
    # TODO 21, phần (3) của đề: tính trung bình mỗi phía, làm tròn 2 chữ số.
    #   Phía nào không có lượt đánh giá nào thì ghi "N/A".
    #   Cẩn thận: phim Psycho (1960) không có lượt nào của nam. Chia thẳng sẽ ném
    #   ZeroDivisionError. Ghi 0.00 cũng sai, vì 0.00 nghĩa là có người chấm 0 điểm,
    #   khác hẳn với không có ai chấm.
    avg_m = "%.2f" % (tong_m / dem_m) if dem_m else "N/A"
    avg_f = "%.2f" % (tong_f / dem_f) if dem_f else "N/A"
    return "%s, %s" % (avg_m, avg_f)


ket_qua_gt = gop.mapValues(dinh_dang) \
                .join(movies) \
                .map(lambda x: (x[1][1][0], x[1][0]))

for ten, s in sorted(ket_qua_gt.collect()):
    print("%-56s %s" % (ten, s))

14
E.T. the Extra-Terrestrial (1982)                        3.81, 3.55
Fight Club (1999)                                        3.50, 3.50
Gladiator (2000)                                         3.59, 3.64
Lawrence of Arabia (1962)                                3.55, 3.31
Mad Max: Fury Road (2015)                                4.00, 3.32
No Country for Old Men (2007)                            3.92, 3.83
Psycho (1960)                                            N/A, 4.00
Sunset Boulevard (1950)                                  4.33, 4.50
The Godfather: Part II (1974)                            4.06, 3.94
The Lord of the Rings: The Fellowship of the Ring (2001) 4.00, 3.80
The Lord of the Rings: The Return of the King (2003)     3.75, 3.90
The Silence of the Lambs (1991)                          3.33, 3.00
The Social Network (2010)                                4.00, 3.67
The Terminator (1984)                                    3.93, 4.14


### Bước 2. Đối chiếu kết quả

Dòng `Psycho (1960)` phải ra `N/A, 4.00`, vì cả 2 lượt đánh giá phim đó đều của
người dùng nữ. Vài dòng khác để đối chiếu nhanh:

```
E.T. the Extra-Terrestrial (1982)   3.81, 3.55
Mad Max: Fury Road (2015)           4.00, 3.32
Psycho (1960)                       N/A, 4.00
Sunset Boulevard (1950)             4.33, 4.50
The Terminator (1984)               3.93, 4.14
```

### Bước 3. Làm lại bằng `broadcast`

Đây là bản Spark của map-side join qua Distributed Cache đã làm ở Lab 1.
Bạn tự viết toàn bộ ô dưới.

In [27]:
# TODO 22: dựng lại da_noi mà không dùng join, bằng broadcast.
#
#   Hai việc cần làm:
#     1. phát bảng giới tính về mọi executor dưới dạng dict UserID -> 'M' hoặc 'F'
#     2. từ ratings, map thẳng sang (MovieID, (Gender, Rating)), tra giới tính
#        trong bảng vừa phát thay vì join
#
bang_gioi_tinh = gioi_tinh.collectAsMap()
bc_gioi_tinh = sc.broadcast(bang_gioi_tinh)
da_noi_2 = ratings.map(lambda r: (r[1], (bc_gioi_tinh.value[r[0]], r[2])))


print(sorted(da_noi.collect()) == sorted(da_noi_2.collect()))   # phải ra True

# users.txt chỉ 50 dòng, vài KB, nên nạp hẳn vào bộ nhớ từng executor rẻ hơn nhiều
# so với shuffle cả hai RDD. Ngưỡng thực tế để cân nhắc broadcast thường là vài
# chục MB. Khi cả hai bên đều lớn thì bắt buộc phải join.

True


### Câu hỏi kiểm tra nhanh, mục D.3

1. Vì sao phải đổi khoá từ UserID sang MovieID giữa chừng, thay vì gộp thẳng một lần?
2. Nếu `users.txt` có 50 triệu dòng thì cách broadcast ở Bước 3 hỏng ở đâu?
3. Sunset Boulevard có 6 lượt của nam và 1 lượt của nữ, điểm lần lượt 4.33 và 4.50.
   Có kết luận được rằng nữ thích phim này hơn nam không?

## D.4. Bài 4 và Bài 5. Nhóm tuổi và nghề nghiệp

**Hai bài cuối, bạn tự viết hoàn toàn.** Cả hai đều là biến thể của D.3.

- **Bài 4:** chia người dùng thành bốn nhóm tuổi `0-18`, `18-35`, `35-50` và `50+`.
  Với mỗi phim, tính điểm trung bình của từng nhóm rồi gộp cả bốn vào một dòng.
- **Bài 5:** tính điểm trung bình và tổng số lượt đánh giá cho từng nghề nghiệp.
  Cột Occupation trong `users.txt` chỉ là mã số, tên nghề nằm ở `occupation.txt`,
  nên phải nối hai lần.

### Bước 1. Viết hàm phân nhóm tuổi

In [28]:
def nhom_tuoi(tuoi):
    """Tách logic phân nhóm ra hàm riêng để đổi ngưỡng cho dễ."""
    if tuoi < 18:
        return "0-18"
    if tuoi < 35:
        return "18-35"
    if tuoi < 50:
        return "35-50"
    return "50+"


NHOM = ["0-18", "18-35", "35-50", "50+"]

# TODO 23: đếm số người dùng trong từng nhóm tuổi.
#   users là (UserID, (Gender, Age, Occupation)), vậy tuổi nằm ở đâu?
phan_bo = users.map(lambda x: (nhom_tuoi(x[1][1]), 1)) \
               .reduceByKey(lambda a, b: a + b)
print(sorted(phan_bo.collect()))
# phải ra [('18-35', 23), ('35-50', 23), ('50+', 4)]

# Nhóm 0-18 không xuất hiện vì người trẻ nhất trong bộ dữ liệu là 19 tuổi.
# Vẫn phải in đủ bốn nhóm theo đúng định dạng đề yêu cầu.

[('18-35', 23), ('35-50', 23), ('50+', 4)]


### Bước 2. Bài 4: tính và in kết quả

Tự viết toàn bộ. Định dạng mỗi dòng: tên phim, rồi bốn nhóm trong một cặp ngoặc vuông.
Nhóm nào không có lượt đánh giá nào thì ghi `N/A`, đúng như cách xử lý ở D.3.

In [29]:
# TODO 24: Bài 4, viết toàn bộ.
#
#   Đường đi gợi ý, bám theo D.3:
#     1. từ users lấy (UserID, nhóm tuổi), dùng hàm nhom_tuoi ở trên
#     2. nối với r_theo_user rồi đổi khoá sang MovieID
#     3. gộp để ra điểm trung bình của từng (phim, nhóm)
#        Gợi ý: dùng khoá ghép (MovieID, nhóm) thì gọn hơn vector 4 số của D.3
#     4. gom bốn nhóm của mỗi phim vào một dòng, thiếu nhóm nào thì ghi N/A
#     5. nối với movies để lấy tên phim rồi in
#
# Kết quả mong đợi, 14 dòng, cột 0-18 luôn N/A. Vài dòng để đối chiếu:
#   Fight Club (1999)        [0-18: N/A, 18-35: 3.50, 35-50: 3.50, 50+: 3.50]
#   Gladiator (2000)         [0-18: N/A, 18-35: 3.43, 35-50: 3.81, 50+: 3.50]
#   Psycho (1960)            [0-18: N/A, 18-35: 4.50, 35-50: 3.50, 50+: N/A]
#   Sunset Boulevard (1950)  [0-18: N/A, 18-35: 4.17, 35-50: 4.50, 50+: 4.50]
#
nhom_user = users.mapValues(lambda u: nhom_tuoi(u[1]))

phim_nhom = r_theo_user.join(nhom_user) \
    .map(lambda x: ((x[1][0][0], x[1][1]), (x[1][0][1], 1))) \
    .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])) \
    .map(lambda x: (x[0][0], (x[0][1], x[1][0] / x[1][1]))) \
    .groupByKey() \
    .mapValues(lambda vs: dict(vs))

ket_qua_tuoi = phim_nhom.join(movies).map(
    lambda x: (x[1][1][0], "[" + ", ".join(
        (f"{n}: {x[1][0][n]:.2f}" if n in x[1][0] else f"{n}: N/A")
        for n in NHOM) + "]"))

for ten, thong_ke in sorted(ket_qua_tuoi.collect()):
    print("%-56s %s" % (ten, thong_ke))

E.T. the Extra-Terrestrial (1982)                        [0-18: N/A, 18-35: 3.56, 35-50: 3.83, 50+: 3.00]
Fight Club (1999)                                        [0-18: N/A, 18-35: 3.50, 35-50: 3.50, 50+: 3.50]
Gladiator (2000)                                         [0-18: N/A, 18-35: 3.43, 35-50: 3.81, 50+: 3.50]
Lawrence of Arabia (1962)                                [0-18: N/A, 18-35: 3.60, 35-50: 3.32, 50+: 3.75]
Mad Max: Fury Road (2015)                                [0-18: N/A, 18-35: 3.36, 35-50: 3.64, 50+: N/A]
No Country for Old Men (2007)                            [0-18: N/A, 18-35: 3.79, 35-50: 3.88, 50+: 4.17]
Psycho (1960)                                            [0-18: N/A, 18-35: 4.50, 35-50: 3.50, 50+: N/A]
Sunset Boulevard (1950)                                  [0-18: N/A, 18-35: 4.17, 35-50: 4.50, 50+: 4.50]
The Godfather: Part II (1974)                            [0-18: N/A, 18-35: 3.78, 35-50: 4.25, 50+: N/A]
The Lord of the Rings: The Fellowship of the Ring

### Bước 3. Bài 5: nối ba RDD

Từ `ratings` lấy `(UserID, Rating)`, nối với `users` để đổi sang mã nghề,
gộp theo mã nghề, cuối cùng nối với `occupations` để lấy tên nghề.
Sắp xếp giảm dần theo số lượt. Tự viết toàn bộ.

In [30]:
# TODO 25: Bài 5, viết toàn bộ.
#
# Kết quả mong đợi, 14 dòng. Vài dòng để đối chiếu:
#   Doctor         TotalRatings: 21, AverageRating: 3.69
#   Engineer       TotalRatings: 18, AverageRating: 3.56
#   Programmer     TotalRatings: 10, AverageRating: 4.25
#   Teacher        TotalRatings:  5, AverageRating: 3.70
#
# Cộng cột TotalRatings phải ra đúng 184. Ba nghề cùng 17 lượt và hai nghề cùng
# 11 lượt nên thứ tự giữa chúng không xác định.
#
# (OccupationID, (tong_diem, so_luot))
tong_theo_nghe = r_theo_user.join(users.mapValues(lambda u: u[2])) \
    .map(lambda x: (x[1][1], (x[1][0][1], 1))) \
    .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))

ket_qua_n = tong_theo_nghe.join(occupations) \
    .map(lambda x: (x[1][1], (x[1][0][1], x[1][0][0] / x[1][0][1])))

for ten, (luot, diem) in sorted(ket_qua_n.collect(), key=lambda x: (-x[1][0], x[0])):
    print("%-16s TotalRatings: %2d, AverageRating: %.2f" % (ten, luot, diem))

print("Tong so luot:", ket_qua_n.map(lambda x: x[1][0]).sum())

Doctor           TotalRatings: 21, AverageRating: 3.69
Engineer         TotalRatings: 18, AverageRating: 3.56
Journalist       TotalRatings: 17, AverageRating: 3.85
Lawyer           TotalRatings: 17, AverageRating: 3.65
Salesperson      TotalRatings: 17, AverageRating: 3.65
Manager          TotalRatings: 16, AverageRating: 3.47
Consultant       TotalRatings: 14, AverageRating: 3.86
Designer         TotalRatings: 13, AverageRating: 4.00
Artist           TotalRatings: 11, AverageRating: 3.73
Nurse            TotalRatings: 11, AverageRating: 3.86
Programmer       TotalRatings: 10, AverageRating: 4.25
Student          TotalRatings:  8, AverageRating: 4.00
Accountant       TotalRatings:  6, AverageRating: 3.58
Teacher          TotalRatings:  5, AverageRating: 3.70
Tong so luot: 184


### Bước 4. Đếm lại số nghề

Chỗ này cho thấy inner join làm mất dữ liệu mà không báo gì.

In [31]:
print(occupations.count())      # phải ra 15
print(ket_qua_n.count())        # phải ra 14

# occupation.txt có 15 nghề nhưng kết quả chỉ ra 14 dòng. Vì sao?
#
# TODO 26: sửa lại để giữ đủ 15 nghề, nghề không ai đánh giá thì ghi 0 lượt.
#   Gợi ý: đổi phép nối cuối cùng sang rightOuterJoin, rồi xử lý giá trị None.
#   Đặt tên RDD kết quả là co_du.
co_du = tong_theo_nghe.rightOuterJoin(occupations)


print(co_du.count())            # phải ra 15
# values() trả về một RDD chứ không phải list, nên phải collect() trước khi duyệt.
print([(ten, 0) for v, ten in co_du.values().collect() if v is None])
# phải ra [('Researcher', 0)]

15
14
15
[('Researcher', 0)]


In [32]:
spark.stop()
print("Da dong SparkSession.")

Da dong SparkSession.


Nhắc lại quy định của môn: cấm `collect()` toàn bộ dữ liệu rồi xử lý bằng pandas
hay Python thuần. Mọi phép gộp, lọc và nối phải làm bằng RDD API.